# Presentation Graphs

This notebook builds graphs from `outputs/` and `outputs_addition/` and saves them as PNG files.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
OUT = ROOT / 'outputs'
OUT_ADD = ROOT / 'outputs_addition'
FIG_DIR = ROOT / 'presentation_graphs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('ggplot')

reports = pd.read_csv(OUT / 'reports_metadata.csv')
operations = pd.read_csv(OUT / 'operations.csv')
equipment = pd.read_csv(OUT / 'equipment_failures.csv')
fluid = pd.read_csv(OUT / 'drilling_fluid.csv')
nds = pd.read_csv(OUT / 'nds_event_matching_results.csv')
bench = pd.read_csv(OUT / 'matching_benchmark.csv')
quality = pd.read_csv(OUT / 'parse_quality_checks.csv')
keywords = pd.read_csv(OUT / 'tfidf_keywords_per_report.csv')

manifest = pd.read_csv(OUT_ADD / 'pdf_additional_tables_manifest.csv')
by_report = pd.read_csv(OUT_ADD / 'pdf_additional_tables_by_report.csv')

print('Loaded outputs and outputs_addition successfully')

In [ ]:
def save_fig(fig, name):
    path = FIG_DIR / name
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)
    print(path.name)


In [ ]:
coverage = pd.Series({
    'Reports': len(reports),
    'Operations': len(operations),
    'Equipment Failures': len(equipment),
    'Drilling Fluid': len(fluid),
    'NDS Matches': len(nds),
}).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
coverage.plot(kind='bar', ax=ax, color=['#1f77b4','#2ca02c','#ff7f0e','#9467bd','#d62728'])
ax.set_title('Pipeline Coverage by Table')
ax.set_ylabel('Row Count')
ax.set_xlabel('')
for i, v in enumerate(coverage.values):
    ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
save_fig(fig, '01_pipeline_coverage.png')


In [ ]:
q = quality.copy()
q['severity'] = q['severity'].fillna('unknown').astype(str).str.lower()
sev = q.groupby(['table_name', 'severity']).size().unstack(fill_value=0)
if not sev.empty:
    sev = sev.sort_values(by=list(sev.columns), ascending=False).head(10)
fig, ax = plt.subplots(figsize=(11, 6))
if not sev.empty:
    sev.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Quality Checks by Table and Severity')
ax.set_ylabel('Count')
ax.set_xlabel('Table')
ax.legend(title='Severity', bbox_to_anchor=(1.02, 1), loc='upper left')
save_fig(fig, '02_quality_severity_stacked.png')


In [ ]:
vals = nds['matched'].astype(str).str.strip().str.lower().map({'true': 'Matched', 'false': 'Unmatched'}).fillna('Unmatched')
counts = vals.value_counts().reindex(['Matched', 'Unmatched']).fillna(0)
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90, colors=['#2ca02c', '#d62728'], wedgeprops={'width': 0.45})
ax.set_title('NDS Matching Success Rate')
save_fig(fig, '03_nds_match_donut.png')


In [ ]:
scores = pd.to_numeric(bench['ensemble_score'], errors='coerce').dropna()
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(scores, bins=20, color='#1f77b4', edgecolor='white')
ax.set_title('Distribution of Ensemble Scores')
ax.set_xlabel('Ensemble Score')
ax.set_ylabel('Frequency')
save_fig(fig, '04_ensemble_score_histogram.png')


In [ ]:
metric_cols = ['score_tfidf_word','score_tfidf_char','score_fuzzy','score_bm25','score_keyword_overlap']
metric_df = bench[metric_cols].apply(pd.to_numeric, errors='coerce')
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot([metric_df[c].dropna().values for c in metric_cols], labels=metric_cols, showfliers=False)
ax.set_title('NDS Matching Methods: Score Distribution')
ax.set_ylabel('Score')
ax.set_xticklabels(metric_cols, rotation=20, ha='right')
save_fig(fig, '05_benchmark_methods_boxplot.png')


In [ ]:
ops_with_well = operations.merge(reports[['report_id', 'wellbore_id']], on='report_id', how='left')
well_counts = ops_with_well['wellbore_id'].fillna('UNKNOWN').value_counts().head(12).sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(well_counts.index, well_counts.values, color='#17becf')
ax.set_title('Top Wells by Operation Count')
ax.set_xlabel('Operations')
for i, v in enumerate(well_counts.values):
    ax.text(v, i, f' {int(v)}', va='center')
save_fig(fig, '06_top_wells_operations_barh.png')


In [ ]:
act_counts = operations['activity_label'].fillna('UNKNOWN').astype(str).value_counts().head(12)
fig, ax = plt.subplots(figsize=(10, 6))
act_counts.plot(kind='bar', ax=ax, color='#ff9896')
ax.set_title('Operation Activity Label Distribution')
ax.set_xlabel('Activity Label')
ax.set_ylabel('Count')
ax.set_xticklabels(act_counts.index, rotation=30, ha='right')
for i, v in enumerate(act_counts.values):
    ax.text(i, v, str(int(v)), ha='center', va='bottom', fontsize=8)
save_fig(fig, '07_activity_label_distribution.png')


In [ ]:
kw = keywords.copy()
kw['tfidf_score'] = pd.to_numeric(kw['tfidf_score'], errors='coerce').fillna(0.0)
kw_top = kw.groupby('keyword', as_index=False)['tfidf_score'].mean().sort_values('tfidf_score', ascending=False).head(15).sort_values('tfidf_score', ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(kw_top['keyword'], kw_top['tfidf_score'], color='#bcbd22')
ax.set_title('Top TF-IDF Keywords (Mean Score Across Reports)')
ax.set_xlabel('Mean TF-IDF Score')
save_fig(fig, '08_top_tfidf_keywords_barh.png')


In [ ]:
man = manifest.sort_values('rows', ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(man['table_name'], man['rows'], color='#8c564b')
ax.set_title('Additional Table Families (outputs_addition)')
ax.set_xlabel('Rows')
save_fig(fig, '09_outputs_addition_table_rows.png')


In [ ]:
def table_count_from_json(x):
    if not isinstance(x, str) or not x.strip():
        return 0
    try:
        obj = json.loads(x)
        return len(obj) if isinstance(obj, dict) else 0
    except Exception:
        return 0

tmp = by_report.copy()
tmp['table_count'] = tmp['tables_json'].apply(table_count_from_json)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(tmp['table_count'], bins=range(0, int(tmp['table_count'].max()) + 2), color='#9467bd', edgecolor='white', align='left')
ax.set_title('Number of Additional Table Types per Report')
ax.set_xlabel('Additional Table Type Count')
ax.set_ylabel('Number of Reports')
save_fig(fig, '10_additional_table_types_per_report_hist.png')


In [ ]:
print('Saved files:')
for p in sorted(FIG_DIR.glob('*.png')):
    print('-', p.name)